In [11]:
!pip install -q langchain langchain-community langchain-huggingface pymupdf faiss-cpu sentence-transformers transformers accelerate bitsandbytes gradio

In [3]:
from google.colab import files

uploaded = files.upload()
pdf_path = next(iter(uploaded))
print("Uploaded file:", pdf_path)

Saving lecturenotes.pdf to lecturenotes.pdf
Uploaded file: lecturenotes.pdf


In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load PDF pages using PyMuPDF
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()
print(f"Loaded {len(documents)} pages.")

# 2. Split pages into overlapping chunks while preserving page metadata
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")

# 3. Embed chunks using Sentence-Transformers
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Store embeddings in FAISS vector store
vector_db = FAISS.from_documents(chunks, embedding_model)
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

print("FAISS vector database indexed successfully!")

/tmp/ipykernel_609/1510186768.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Loaded 86 pages.
Split into 422 chunks.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS vector database indexed successfully!


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.2,
    do_sample=True,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_gen_pipeline)

def format_docs(docs):
    formatted = []
    for doc in docs:
        page_num = doc.metadata.get("page", "N/A")
        # PyMuPDF page index is 0-based; display 1-based page number too
        page_label = page_num + 1 if isinstance(page_num, int) else page_num
        formatted.append(f"[PDF Page {page_label}]\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)

prompt_template = """<|system|>
You are a helpful Machine Learning teaching assistant. Answer the user's question using ONLY the provided lecture note context. Cite the PDF page number(s) when possible. If the answer is not found in the provided context, reply: "I don't have enough information in the provided lecture notes to answer that."
</s>
<|user|>
Context:
{context}

Question:
{question}
</s>
<|assistant|>"""

prompt = PromptTemplate.from_template(prompt_template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Assistant ready!")

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

RAG Assistant ready!


In [9]:
import re

def clean_query(text: str) -> str:
    # Collapse 3+ repeated letters first, then fix common double-first-letter typos like "mmachine"
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\b([a-zA-Z])\1([a-zA-Z]{3,})\b", r"\1\2", text)
    return text.strip()

questions = [
    "What is the difference between explicit and implicit programming in machine learning?",
    "How does k-means clustering work, and why is it an iterative algorithm?",
    "Why can a single perceptron not learn the XOR function?",
    "What is the boiling point of water at sea level?",
    "What is mmachine learning?"
]

for idx, q in enumerate(questions, start=1):
    cleaned_q = clean_query(q)
    print(f"=== Sample Question {idx} ===")
    print(f"Q : {q}")
    if cleaned_q != q:
        print(f"Q : {cleaned_q}")
    response = rag_chain.invoke(cleaned_q).strip()
    print(f"A: {response}\n")

=== Sample Question 1 ===
Q : What is the difference between explicit and implicit programming in machine learning?
A: In explicit programming in machine learning, the algorithm for solving a specific task, such as sorting an array or calculating a payroll, is designed and implemented by the programmer. This approach requires a deep understanding of the problem and the ability to devise a specific algorithm to solve it.

In contrast, implicit programming in machine learning involves defining an objective function and a classifier with adjustable parameters, such as a neural network, and training it on a labeled dataset. The algorithm is not explicitly designed, but rather learned through the training process. This approach is more suitable for complex tasks where it is difficult to design algorithms, such as image classification or outlier detection.

The main difference between explicit and implicit programming is the level of human intervention in designing the algorithm. In explicit

In [10]:
import re
import gradio as gr

def clean_query(text: str) -> str:
    # Collapse 3+ repeated letters first, then fix common double-first-letter typos like "mmachine"
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\b([a-zA-Z])\1([a-zA-Z]{3,})\b", r"\1\2", text)
    return text.strip()

def query_lecture_notes(question):
    if not question or not question.strip():
        return "Please enter a question about the Machine Learning lecture notes."
    cleaned_q = clean_query(question.strip())
    return rag_chain.invoke(cleaned_q).strip()

demo = gr.Interface(
    fn=query_lecture_notes,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask a question about lecturenotes.pdf...",
        label="Your Question"
    ),
    outputs=gr.Textbox(
        lines=8,
        label="Answer"
    ),
    title="ML Lecture Notes RAG Assistant",
    description="Grounded answers from lecturenotes.pdf using FAISS, Sentence-Transformers, and Zephyr-7B."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://df4a8551508967c4e7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
